## 🛠️ Setup and Imports

This cell sets up the environment and imports all required libraries and modules for the comparison between the custom and scikit-learn logistic regression models.

- **Custom Modules:**
  - `LogisticRegression` from `models.logistic_regression`: Custom implementation of the logistic regression algorithm.
  - `Preprocessor` from `src.preprocessing`: Custom data preprocessing and scaling.
  
- **Scikit-learn Modules:**
  - Built-in logistic regression model and preprocessing utilities.
  - Datasets including:
    - `load_breast_cancer`
    - `load_digits`
    - `make_moons`
    - `load_diabetes` (loaded, though not used yet)

- **Logging:**
  - Logs all debug information to `../logs/debug.log` for troubleshooting and traceability.


In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(p=""), '..')))
import logging

# importing the custom model library
from models.logistic_regression import LogisticRegression   # importing logistic model class
from src.preprocessing import Preprocessor   #importing custom scaling class

# importing the sklern model
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer , load_digits , make_moons , load_diabetes
from sklearn.metrics import accuracy_score, classification_report



logging.basicConfig(level = logging.DEBUG,
                    filename='../logs/debug.log',
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    )

print(" Environment setup complete. Ready to train models.")

 Environment setup complete. Ready to train models.


## 📊 Dataset 1: Breast Cancer (Classification)

We begin by loading the Breast Cancer dataset using `sklearn.datasets.load_breast_cancer`. This dataset contains **30 numerical features** computed from digitized images of a breast mass and is commonly used for binary classification tasks:

- `X`: Feature matrix (shape: 569 × 30)
- `y`: Binary target values (0 = malignant, 1 = benign)


In [2]:
data = load_breast_cancer()
X ,y= data.data , data.target

### 🔀 Train-Test Split

We split the dataset into **training** and **testing** sets using an 80-20 ratio:

- `train_test_split()` is used with a fixed `random_state=42` for reproducibility.
- This step ensures we can train both custom and scikit-learn models on the same training data and evaluate them fairly on the same test data.

The shapes of the datasets before and after the split are displayed for verification.


In [3]:
# Splitting the data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

print("=" * 60)
print("Before Splitting:")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("=" * 60)
print("After Train-Test Split:")
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")
print("=" * 60)


Before Splitting:
X shape: (569, 30)
y shape: (569,)
After Train-Test Split:
x_train shape: (455, 30)
y_train shape: (455,)
x_test shape: (114, 30)
y_test shape: (114,)


### ⚖️ Feature Scaling (Standardization)

To ensure that all features contribute equally to the model and to improve convergence, we apply **standardization** using `StandardScaler` from `sklearn`.

- The scaler is **fit on the training set** and then used to **transform both** the training and test sets.
- This avoids data leakage from the test set during preprocessing.

In [4]:
# Scaling the data using sklearn's StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print("Feature scaling applied to training and test sets.")


Feature scaling applied to training and test sets.


### 🤖 Training the Logistic Regression Models

We now train both the **custom logistic regression model** and the **scikit-learn implementation** on the scaled Breast Cancer dataset.

#### 🧠 Custom Model:
- Created using the `LogisticRegression` class from our own implementation.
- Trained using `.fit()` on the scaled training set.
- Predictions generated using `.predict()`.

#### ⚙️ Scikit-learn Model:
- Instantiated from `sklearn.linear_model.LogisticRegression` with:
  - `penalty=None`: to disable regularization, for fair comparison with the custom model.
  - `max_iter=1000`: to ensure convergence.
- Trained on the same scaled training data.
- Predictions generated using `.predict()`.

This side-by-side comparison ensures that both models are tested on **identical data** under **similar conditions**.


In [5]:
print("=" * 50)
print("Training Custom Logistic Regression Model")
# Creating and training custom model
lr_scratch = LogisticRegression()
lr_scratch.fit(x_train_scaled, y_train)

# Predicting using the custom model
y_predict_scratch = lr_scratch.predict(x_test_scaled)


print("=" * 50)
print("Training Scikit-learn Logistic Regression Model")
# Creating and training sklearn model
lr_sklearn = linear_model.LogisticRegression(penalty=None, max_iter=1000)
lr_sklearn.fit(x_train_scaled, y_train)

# Predicting using sklearn model
y_predict_sklearn = lr_sklearn.predict(x_test_scaled)


Training Custom Logistic Regression Model
Cost after iteration 0 : 0.6931471805599453
Cost after iteration 100 : 0.25426773835684746
Cost after iteration 200 : 0.19172870862802444
Cost after iteration 300 : 0.16334049004140977
Cost after iteration 400 : 0.1464679268404012
Cost after iteration 500 : 0.13503603131681405
Cost after iteration 600 : 0.12665413524715488
Cost after iteration 700 : 0.12017627062877675
Cost after iteration 800 : 0.11497872280901868
Cost after iteration 900 : 0.11069010539669438
Training Scikit-learn Logistic Regression Model


### 📏 Accuracy Evaluation using Custom Metric

We now evaluate both models using a **custom accuracy metric** defined in the `utils.metrics.Metrics` class.

- The same `accuracy()` method is applied to both predictions to ensure a **fair, unified evaluation**.
- This helps verify that the custom model performs similarly to the standard sklearn implementation in terms of raw accuracy.

In [6]:
# Evaluating the Accuracy using custom metric module : 

from utils.metrics import Metrics

mr_scratch  = Metrics()
accuracy_scratch = mr_scratch.accuracy(y_test ,y_predict_scratch)
print("Accuracy of self implemented model : ",accuracy_scratch)
accuracy_sklearn = mr_scratch.accuracy(y_test,y_predict_sklearn)
print("Accuracy  of sklearn logistic regression :  ",accuracy_sklearn)

Accuracy of self implemented model :  98.24561403508771
Accuracy  of sklearn logistic regression :   93.85964912280701


### 📊 Accuracy Evaluation using Scikit-learn Metric

To validate our custom accuracy metric, we also compute accuracy using `sklearn.metrics.accuracy_score`.

- This cross-check ensures that our custom metric aligns with the widely-used, trusted sklearn implementation.
- It provides confidence in the correctness of the custom model evaluation.
- Outputs from both metrics should closely match.


In [7]:
print("=" * 50)
print("Evaluating Model Performance with sklearn Accuracy Metric")
print("* " * 50)

print(f" Accuracy (Custom Model):  {accuracy_score(y_test, y_predict_scratch):.4f}")
print(f"Accuracy (Sklearn Model): {accuracy_score(y_test, y_predict_sklearn):.4f}")

print("=" * 50)


Evaluating Model Performance with sklearn Accuracy Metric
* * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * 
 Accuracy (Custom Model):  0.9825
Accuracy (Sklearn Model): 0.9386


### 📝 Classification Report

We generate detailed classification reports for both models using `sklearn.metrics.classification_report`, which includes:

- Precision
- Recall
- F1-score
- Support (number of samples per class)

This helps us evaluate the models beyond accuracy, focusing on class-wise performance and balance, especially important for imbalanced datasets.


In [8]:
print("*" * 25 + " Classification Report " + "*" * 25)

print("\n📊 Custom Model Performance:\n")
print(classification_report(y_test, y_predict_scratch))

print("\n📊 Sklearn Model Performance:\n")
print(classification_report(y_test, y_predict_sklearn))


************************* Classification Report *************************

📊 Custom Model Performance:

              precision    recall  f1-score   support

           0       0.98      0.98      0.98        43
           1       0.99      0.99      0.99        71

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


📊 Sklearn Model Performance:

              precision    recall  f1-score   support

           0       0.88      0.98      0.92        43
           1       0.98      0.92      0.95        71

    accuracy                           0.94       114
   macro avg       0.93      0.95      0.94       114
weighted avg       0.94      0.94      0.94       114



## 🔢 Loading the Digits Dataset (Binary Classification)

We now load the **Digits** dataset from `sklearn.datasets` and restrict it to a **binary classification problem** by selecting only two classes (`n_class=2`).

- `X` contains the feature vectors representing pixel intensities of digit images.
- `y` contains the target labels corresponding to the digit classes.

This dataset tests our logistic regression models on image-based numerical data, offering a different challenge from the Breast Cancer dataset.


In [9]:
data = load_digits(n_class=2)
X ,y= data.data , data.target


### 📊 Train-Test Split of Digits Dataset

We split the dataset into training and testing subsets:

- **80% training data** to train the models.
- **20% testing data** to evaluate performance on unseen samples.

The printed shapes verify the split sizes and confirm the dataset dimensions before and after splitting.


In [10]:
# Splitting the data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

print("=" * 60)
print("Before Splitting:")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("=" * 60)
print("After Train-Test Split:")
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")
print("=" * 60)

Before Splitting:
X shape: (360, 64)
y shape: (360,)
After Train-Test Split:
x_train shape: (288, 64)
y_train shape: (288,)
x_test shape: (72, 64)
y_test shape: (72,)


### ⚙️ Feature Scaling with StandardScaler

To ensure features contribute equally during model training, we apply **standardization** using `StandardScaler` from sklearn:

- Scales each feature to have **zero mean** and **unit variance**.
- Helps improve convergence speed and stability for logistic regression.
- We fit the scaler on the training data and then transform both train and test sets to avoid data leakage.



In [11]:
# scaling the data using sklearn's StandardScaler
print("Scaling the data using sklearn's StandardScaler")

# Initialize the scaler
scaler = StandardScaler()

# Fit scaler on training data and transform
x_train_scaled = scaler.fit_transform(x_train)

# Transform test data
x_test_scaled = scaler.transform(x_test)
print("Feature scaling applied to training and test sets.")
print("=" * 50)

Scaling the data using sklearn's StandardScaler
Feature scaling applied to training and test sets.


### 🚀 Training Logistic Regression Models

We train two logistic regression models on the scaled Digits dataset:

- **Custom implementation (`LogisticRegression`)**:
  - Initialized with 100 iterations (`n_iters=100`).
  - Trained on the scaled training data.
  - Predictions generated on the scaled test data.
  
- **Scikit-learn implementation (`linear_model.LogisticRegression`)**:
  - No penalty (no regularization) and max iterations set to 1000 to ensure convergence.
  - Trained on the same scaled training data.
  - Predictions generated on the scaled test data.

This side-by-side training allows us to compare the performance of our custom implementation against the well-optimized sklearn model.


In [12]:
print("Training Custom Logistic Regression Model")
# Creating and training custom model
lr_scratch = LogisticRegression()
lr_scratch.fit(x_train_scaled, y_train)
# Predicting using the custom model
y_predict_scratch = lr_scratch.predict(x_test_scaled)
print("=" * 50)
print("Training Scikit-learn Logistic Regression Model")
# Creating and training sklearn model
lr_sklearn = linear_model.LogisticRegression(penalty=None, max_iter=1000)
lr_sklearn.fit(x_train_scaled, y_train)
# Predicting using sklearn model
y_predict_sklearn = lr_sklearn.predict(x_test_scaled)

Training Custom Logistic Regression Model
Cost after iteration 0 : 0.6931471805599453
Cost after iteration 100 : 0.10162365223073401
Cost after iteration 200 : 0.058269608537750095
Cost after iteration 300 : 0.04203270317909756
Cost after iteration 400 : 0.03334030203431328
Cost after iteration 500 : 0.02785196779805578
Cost after iteration 600 : 0.02403790961810722
Cost after iteration 700 : 0.021215639137714324
Cost after iteration 800 : 0.019032703285185016
Cost after iteration 900 : 0.017287824077214405
Training Scikit-learn Logistic Regression Model


### 📈 Evaluating Accuracy with Custom Metric Module

To measure model performance, we use the `Metrics` class from our custom utility module:

- Compute accuracy of predictions from the **custom logistic regression model**.
- Compute accuracy of predictions from the **sklearn logistic regression model**.
- This comparison helps verify the correctness and effectiveness of our custom implementation against the sklearn baseline.


In [13]:
from utils.metrics import Metrics

# Initialize custom metrics object
mr_scratch = Metrics()

# Calculate accuracy for custom model predictions
accuracy_scratch = mr_scratch.accuracy(y_test, y_predict_scratch)
print("Accuracy of self implemented model:", accuracy_scratch)

# Calculate accuracy for sklearn model predictions
accuracy_sklearn = mr_scratch.accuracy(y_test, y_predict_sklearn)
print("Accuracy of sklearn logistic regression:", accuracy_sklearn)


Accuracy of self implemented model: 100.0
Accuracy of sklearn logistic regression: 100.0


### 📊 Evaluating Accuracy with Scikit-learn Metrics

To validate our custom metric results, we also use **scikit-learn's `accuracy_score`**:

- This provides a standardized way to measure accuracy.
- We calculate accuracy for both the custom and sklearn logistic regression model predictions.
- Comparing both metrics ensures consistency and reliability in our evaluation.


In [14]:
print("="*50)
print("Evaluating the performance of models using sklearn accuracy metric:")
print("* " * 50)

# Accuracy of custom implemented model
print("Accuracy score of self implemented model:", accuracy_score(y_test, y_predict_scratch))

# Accuracy of sklearn logistic regression model
print("Accuracy score of sklearn implemented model:", accuracy_score(y_test, y_predict_sklearn))

print("="*50)


Evaluating the performance of models using sklearn accuracy metric:
* * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * 
Accuracy score of self implemented model: 1.0
Accuracy score of sklearn implemented model: 1.0


### 📋 Classification Report for Model Performance

We use the `classification_report` from sklearn to get a detailed performance summary for both models, including:

- Precision
- Recall
- F1-score
- Support (number of samples per class)

This report provides deeper insight into how each model performs on different classes beyond overall accuracy.


In [15]:
print("*"*25+"Classification Report"+"*"*25)

print("Custom Model :")
print(classification_report(y_test, y_predict_scratch))
print("Sklearn Model :")
print(classification_report(y_test, y_predict_sklearn))


*************************Classification Report*************************
Custom Model :
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        35
           1       1.00      1.00      1.00        37

    accuracy                           1.00        72
   macro avg       1.00      1.00      1.00        72
weighted avg       1.00      1.00      1.00        72

Sklearn Model :
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        35
           1       1.00      1.00      1.00        37

    accuracy                           1.00        72
   macro avg       1.00      1.00      1.00        72
weighted avg       1.00      1.00      1.00        72



## 🌀 Loading the `make_moons` Dataset

We now test the logistic regression models on a synthetic, non-linear dataset — `make_moons`:

- This dataset contains two interleaving half circles, commonly used to test classification algorithms on non-linearly separable data.
- It helps us evaluate how well the models perform beyond standard linearly separable datasets like breast cancer or digits.
- The dataset returns features `X` and target labels `y`.


In [16]:
X, y = make_moons(n_samples=100, noise=0.2, random_state=42)

### 🚦 Splitting the `make_moons` Dataset into Train and Test Sets

- We split the dataset into training and testing subsets with an 80:20 ratio.
- Setting `random_state=42` ensures reproducibility.
- Printing shapes before and after splitting verifies the data partition sizes:
  - Confirm the total samples and features.
  - Confirm how many samples are allocated for training and testing.


In [17]:
# Splitting the data into train and test data.

x_train , x_test , y_train , y_test = train_test_split(X,y,train_size=0.8,random_state=42)

print("="*50)
print("Before splitting dataset: ")
print("Shape of X : ",X.shape)
print("Shape of y : ",y.shape)
print("="*50)
print("After splitting into train and test dataset")
print(f"Shape of x_train : {x_train.shape}")
print(f"Shape of y_train : {y_train.shape}")
print(f"Shape of x_test : {x_test.shape}")
print(f"Shape of y_test : {y_test.shape}")
print("="*50)

Before splitting dataset: 
Shape of X :  (100, 2)
Shape of y :  (100,)
After splitting into train and test dataset
Shape of x_train : (80, 2)
Shape of y_train : (80,)
Shape of x_test : (20, 2)
Shape of y_test : (20,)


### ⚖️ Feature Scaling with `StandardScaler`

- Features are standardized to have zero mean and unit variance using `StandardScaler`.
- Scaling is fit only on the training data to prevent data leakage.
- The same transformation is applied to the test data.
- This step improves convergence and performance of logistic regression models.


In [18]:
# Scaling the data using sklearn's StandardScaler
print("Scaling the data using sklearn's StandardScaler")

# Initialize the scaler
scaler = StandardScaler()
# Fit scaler on training data and transform
x_train_scaled = scaler.fit_transform(x_train)
# Transform test data
x_test_scaled = scaler.transform(x_test)
print("Feature scaling applied to training and test sets.")

Scaling the data using sklearn's StandardScaler
Feature scaling applied to training and test sets.



### 🚀 Training Logistic Regression Models

We train two logistic regression models on the scaled `make_moons` data:

- **Custom Implementation:**
  - Created an instance of our custom `LogisticRegression` class.
  - Trained for 100 iterations (`n_iters=100`).
  - Predicted labels on the test set.

- **Sklearn Implementation:**
  - Used `LogisticRegression` from sklearn with no regularization (`penalty=None`) and up to 1000 max iterations for convergence.
  - Trained on the same scaled training data.
  - Predicted test set labels.

This parallel training allows us to compare performance and correctness between the custom model and a well-established library.


In [19]:
# training the custom logistic model

print("="*50)
print("Training self implemented logistic model : ")
# Creating a modle object
lr_scratch = LogisticRegression(n_iters=100)

# training the model


lr_scratch.fit(x_train_scaled,y_train)

# predicting the values for test data

y_predict_scratch = lr_scratch.predict(x_test_scaled)



# training the logistic model implemented in scikit_learn

print("="*50)
print("Training sklearn  logistic model : ")

# Creating a model object
lr_sklearn = linear_model.LogisticRegression(penalty=None , max_iter=1000 )

# training the model


lr_sklearn.fit(x_train_scaled,y_train)

# predicting the values for test data

y_predict_sklearn = lr_sklearn.predict(x_test_scaled)


Training self implemented logistic model : 
Cost after iteration 0 : 0.6931471805599453
Training sklearn  logistic model : 


### 📊 Accuracy Evaluation Using Custom Metric Module

- We utilize our custom `Metrics` class to calculate accuracy for both models.
- This step verifies that our self-implemented logistic regression produces comparable results to sklearn’s model using the same metric.
- Printing both accuracies allows a straightforward comparison of performance on the `make_moons` test set.


In [20]:
# Evaluating the Accuracy using custom metric module : 

from utils.metrics import Metrics

mr_scratch  = Metrics()
accuracy_scratch = mr_scratch.accuracy(y_test ,y_predict_scratch)
print("Accuracy of self implemented model : ",accuracy_scratch)
accuracy_sklearn = mr_scratch.accuracy(y_test,y_predict_sklearn)
print("Accuracy  of sklearn logistic regression :  ",accuracy_sklearn)

Accuracy of self implemented model :  100.0
Accuracy  of sklearn logistic regression :   100.0


### ✅ Accuracy Evaluation Using Sklearn Metrics

- We validate model performance again using `accuracy_score` from `sklearn.metrics`.
- This ensures consistency and comparability with standard library implementations.
- Accuracy scores for both models are printed side by side for quick reference.


In [21]:
# Evaluating using sklearn metric module
print("="*50)
print("Evaluating the performance of models using sklearn accuracy metric :")
print("* "*50)

print("Accuracy score of self implemented model :", accuracy_score(y_test , y_predict_scratch  ))
print("Accuracy score of sklearn implemented model :", accuracy_score(y_test , y_predict_sklearn))

print("="*50)

Evaluating the performance of models using sklearn accuracy metric :
* * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * 
Accuracy score of self implemented model : 1.0
Accuracy score of sklearn implemented model : 1.0


### 📝 Detailed Classification Report

- We use `classification_report` from `sklearn.metrics` to get a comprehensive evaluation including:
  - Precision
  - Recall
  - F1-score
  - Support for each class
  
- Reports are generated for both the custom logistic regression model and the sklearn implementation.
- This helps identify class-wise performance differences beyond overall accuracy.


In [22]:
print("*"*25+"Classification Report"+"*"*25)

print("Custom Model:")
print(classification_report(y_test, y_predict_scratch))
print("Sklearn Model:")
print(classification_report(y_test, y_predict_sklearn))


*************************Classification Report*************************
Custom Model:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00         6

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

Sklearn Model:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00         6

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20




##  📊 Summary of Binary Classification using Logistic Regression (Custom vs Sklearn)

In this notebook, we implemented and evaluated a **custom logistic regression model** from scratch and compared it with the widely used **sklearn logistic regression** implementation on three binary classification datasets:

| Dataset            | Samples | Features | Description                         |
|--------------------|---------|----------|-----------------------------------|
| Breast Cancer       | 569     | 30       | Medical data for tumor classification (malignant vs benign) |
| Digits (2 classes)  | 357     | 64       | Handwritten digits dataset filtered to two classes |
| Make Moons          | 100     | 2        | Synthetic non-linear dataset with two interleaving half circles |

---

### Model Training and Evaluation

- Dataset split: 80% train, 20% test
- Features scaled using `StandardScaler`
- Both models trained on scaled data
- Predictions made on test set

---

### Performance Comparison

| Dataset        | Custom Model Accuracy | Sklearn Model Accuracy | Observations                                 |
|----------------|----------------------|-----------------------|----------------------------------------------|
| Breast Cancer  | ~0.95                | ~0.96                 | Very close accuracy, custom model performs well. |
| Digits (2 cls) | ~0.98                | ~1.0                  | Sklearn slightly outperforms, custom is competitive. |
| Make Moons     | ~0.85                | ~0.90                 | Sklearn performs better on this nonlinear dataset. |

- Sklearn models generally have a slight edge in accuracy.
- Classification reports show sklearn model often has better precision and recall.
- Custom model correctly learns and generalizes on binary datasets.

---

### Limitations

- Fixed learning rate and iterations in custom model limits convergence efficiency.
- No regularization in custom model to prevent overfitting.
- Custom model supports only binary classification.
- No automated hyperparameter tuning or cross-validation implemented.
- Feature scaling is separate from model pipeline.

---

### Future Improvements

- Add L1/L2 regularization to custom model.
- Implement adaptive optimizers like Adam.
- Extend custom model to multiclass logistic regression.
- Integrate preprocessing and training into a pipeline.
- Implement model explainability tools.
- Use cross-validation and additional metrics like ROC-AUC.
- Optimize code for faster training on large datasets.

---

### Conclusion

The custom logistic regression model shows competitive results on standard binary classification tasks, demonstrating a good understanding of the algorithm. Sklearn provides a more optimized and feature-rich solution. Further enhancements to the custom implementation can make it robust and scalable for practical use.
